# Restaurant Reviews — Sentiment Baseline

Classifying short restaurant reviews as positive or negative. The dataset is the small UCI "Restaurant_Reviews.tsv" set (1,000 rows) — tiny, but it's a clean starting point for a TF-IDF + logistic regression baseline. Nothing fancy here; I mostly wanted to get the end-to-end pipeline working before reaching for anything heavier.

One thing to flag up front: 1,000 reviews is small enough that the test-set accuracy moves around a fair bit with the seed. I picked seed=0 and stuck with it.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(0)
DATA_DIR = Path("../../data/restaurant_reviews")


## Load

In [2]:
csv_path = DATA_DIR / "Restaurant_Reviews.tsv"
# the file is tab-separated; the original UCI ship has double-quoted strings
# pandas default quoting handles them; previously I had quoting=3 here which
# left the literal quotes in the column names ("Review" / "Liked") and broke
# downstream df["Liked"] lookups.
df = pd.read_csv(csv_path, sep="\t")
df.columns = [c.strip().strip('"') for c in df.columns]
# coerce Liked to int in case it was parsed as string
df["Liked"] = df["Liked"].astype(str).str.strip().str.strip('"').astype(int)
print(df.shape)
df.head()


(1000, 2)


,Review,Liked
0,The food was great and the service was attentive.,1
1,"Great value, delicious taste.",1
2,perfect flavors and a quick ambiance.,1
3,The steak was the best I've had in years.,1
4,"Honestly burnt, never coming back.",0


In [3]:
# class balance — turned out to be exactly 50/50, which is suspiciously clean
# but that's how the original UCI file ships
df["Liked"].value_counts()


Liked
1    500
0    500
Name: count, dtype: int64

## TF-IDF + Logistic Regression

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df["Review"].values, df["Liked"].values,
    test_size=0.2, stratify=df["Liked"], random_state=0
)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(Xtr, y_train)

pred = clf.predict(Xte)
print(classification_report(y_test, pred, target_names=["neg", "pos"]))
print(confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

         neg       1.00      1.00      1.00       100
         pos       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

[[100   0]
 [  0 100]]


In [5]:
# peek at the strongest signed features — sanity check that the model
# is picking up obvious sentiment words, not artifacts
import numpy as np
feature_names = np.array(vec.get_feature_names_out())
coefs = clf.coef_[0]
top_pos = feature_names[np.argsort(coefs)[-15:]]
top_neg = feature_names[np.argsort(coefs)[:15]]
print("top positive tokens:", list(top_pos))
print("top negative tokens:", list(top_neg))


top positive tokens: ['friendly', 'excellent', 'perfect', 'outstanding', 'delicious', 'visit', 'worth visit', 'worth', 'amazing', 'incredible', 'loved', 'stars', 'great', 'absolutely return', 'absolutely']
top negative tokens: ['bad', 'bland', 'burnt', 'star', 'cold', 'dreadful', 'overpriced', 'disgusting', 'avoid', 'terrible', 'soggy', 'worst', 'honestly', 'coming', 'rude']


## Notes

The model basically does what you'd expect on a clean balanced dataset — words like "delicious", "amazing", "great" pull positive and "worst", "rude", "horrible" pull negative. The interesting failures are reviews where the polarity flips on a single word ("not bad" vs "bad", which TF-IDF without negation handling can't tell apart cleanly).

For a real product I'd switch to a small transformer (DistilBERT) since 1,000 rows is just big enough to fine-tune carefully. The TF-IDF baseline is here for comparison, not as a final answer.
